In [1]:
import typing

import torch
import linear_operator

In [2]:
NOISE: typing.Final[float] = 1e-6
K: typing.Final[int] = 10


def distance_matrix(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
	r"""To compute the pairwise distance matrix between two sets of vectors

	Parameters
	----------
	x : torch.Tensor
		The first set of vectors
	y : torch.Tensor
		The second set of vectors

	Returns
	-------
	torch.Tensor
		The pairwise distance matrix
	"""
	return torch.linalg.norm(x[:, torch.newaxis] - y[torch.newaxis], dim=-1)


def knn(train_points: torch.Tensor, k: int, test_points: torch.Tensor | None = None) -> torch.Tensor:
	r"""To find the k-nearest neighbors of a set of points

	Author: Josue N Rivera (github.com/wzjoriv)

	Date: 7/3/2021

	Description: Snippet of various clustering implementations only using PyTorch

	Full project repository: https://github.com/wzjoriv/Lign (A graph deep learning framework that works alongside PyTorch)

	Parameters
	----------
	train_points : torch.Tensor
		The training points
	k : int
		The number of neighbors to consider
	test_points : torch.Tensor | None, optional, shape of (N, D)
		The points to predict the labels for, by default None (and uses the training points)

	Returns
	-------
	The k-nearest neighbors average distances shape of (N,)
	"""
	assert train_points.ndim == 2
	if test_points is not None:
		assert test_points.ndim == 2
	else: # test_points is None
		test_points = train_points
	return distance_matrix(test_points, train_points).topk(k, -1, False, False).values.mean()

def cov(lengthscale: torch.Tensor, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
	r"""To calculate the covariance matrix using RBF kernel

	Parameters
	----------
	lengthscale : torch.Tensor
		The characteristic lengths
	x1 : torch.Tensor
		The first feature set
	x2 : torch.Tensor
		The second feature set

	Returns
	-------
	torch.Tensor
		The covariance matrix
	"""
	return torch.exp(-torch.square((x1[:, torch.newaxis] - x2) / lengthscale).sum(-1))


def predict(lengthscale: torch.Tensor, x_test: torch.Tensor, x_train: torch.Tensor, n_pt: int, y: torch.Tensor) -> torch.Tensor:
	nearest_distances: typing.Final[torch.Tensor] = knn(x_train[:n_pt], K)
	noise: typing.Final[torch.Tensor] = NOISE * torch.eye(n_pt) + NOISE * torch.diag(torch.square(nearest_distances / nearest_distances.median()))
	return cov(torch.exp(lengthscale), x_test, x_train[:n_pt]) @ linear_operator.utils.stable_pinverse(cov(torch.exp(lengthscale), x_train, x_train[:n_pt]) + torch.cat((noise, torch.zeros(x_train.shape[0] - n_pt, n_pt)))) @ y

In [3]:
x_train = torch.randn((10, 2)).detach()
n_pt = 5
x_test = torch.randn((3, 2)).detach()
lengthscale = torch.log(torch.tensor((1.0, 0.5))).detach()
y = torch.rand(x_train.shape[0])

In [4]:
def x_test_func(x_test: torch.Tensor):
	print(x_test)
	with torch.no_grad():
		x_test = x_test.detach().requires_grad_()
	for f in (lambda x: x.sum(-1), lambda x: x[:, 0] + 2 * x[:, 1], lambda x: x[:, 0].square() / 2.0 + x[:, 1].exp()):
		print(torch.autograd.grad(f(x_test), x_test, torch.ones(x_test.shape[0]), False, False, True, True, False, True)[0])


x_test_func(x_test)

tensor([[-1.3797e-04,  1.8292e+00],
        [ 1.3403e+00, -7.8053e-01],
        [ 1.0566e+00,  1.1541e-01]])
tensor([[1., 1.],
        [1., 1.],
        [1., 1.]])
tensor([[1., 2.],
        [1., 2.],
        [1., 2.]])
tensor([[-1.3797e-04,  6.2288e+00],
        [ 1.3403e+00,  4.5816e-01],
        [ 1.0566e+00,  1.1223e+00]])


In [5]:
def y_test_func(y: torch.Tensor):
	with torch.no_grad():
		y = y.detach().requires_grad_()
	print(y)
	for f in (lambda x: torch.stack((torch.stack((x.sum(), x[:3].sum() + x.sum(), x.square().sum() / 2.0)), torch.stack((x.sum().exp(), x.exp().sum(), (x[:3, None] * x).sum())))),):
		out = f(y)
		grad = torch.autograd.grad(out.ravel(), y, torch.eye(out.numel()), False, False, True, True, True, True)[0].reshape(out.shape + y.shape)
		print(out.shape, grad.shape)
		print(grad)

y_test_func(y)

tensor([0.6734, 0.5593, 0.9626, 0.9975, 0.1833, 0.9650, 0.6024, 0.8749, 0.4023,
        0.3195], requires_grad=True)
torch.Size([2, 3]) torch.Size([2, 3, 10])
tensor([[[1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
          1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00],
         [2.0000e+00, 2.0000e+00, 2.0000e+00, 1.0000e+00, 1.0000e+00,
          1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00],
         [6.7338e-01, 5.5925e-01, 9.6262e-01, 9.9753e-01, 1.8327e-01,
          9.6502e-01, 6.0235e-01, 8.7486e-01, 4.0227e-01, 3.1946e-01]],

        [[6.9230e+02, 6.9230e+02, 6.9230e+02, 6.9230e+02, 6.9230e+02,
          6.9230e+02, 6.9230e+02, 6.9230e+02, 6.9230e+02, 6.9230e+02],
         [1.9609e+00, 1.7494e+00, 2.6186e+00, 2.7116e+00, 1.2011e+00,
          2.6248e+00, 1.8264e+00, 2.3985e+00, 1.4952e+00, 1.3764e+00],
         [8.7353e+00, 8.7353e+00, 8.7353e+00, 2.1953e+00, 2.1953e+00,
          2.1953e+00, 2.1953e+00, 2.1953e+00, 2.1953e+00, 2.1953

In [6]:
def x_y_test(x: torch.Tensor, y: torch.Tensor):
	with torch.no_grad():
		x = x.detach().requires_grad_()
		y = y.detach().requires_grad_()
	out = torch.stack((x.sum() + y.sum(), x.square().sum() / 2.0 + y[:3].exp().sum()))
	print(
		torch.autograd.grad(out.ravel(), x, torch.eye(out.numel()), False, False, True, True, True, True)[0].reshape(out.shape + x.shape),
		torch.autograd.grad(out.ravel(), y, torch.eye(out.numel()), False, False, True, True, True, True)[0].reshape(out.shape + y.shape),
		sep="\n"
	)

x_y_test(x_train, y)

tensor([[[ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000],
         [ 1.0000,  1.0000]],

        [[ 0.4082,  0.5449],
         [-0.8296, -0.6939],
         [-0.1450, -1.4993],
         [ 1.3379, -0.7213],
         [ 0.3358, -0.9963],
         [ 1.0831, -0.4652],
         [ 0.1802,  0.8315],
         [ 0.3375, -0.2569],
         [-0.9362, -1.5743],
         [ 0.1814,  1.1840]]])
tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [1.9609, 1.7494, 2.6186, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])
